In [52]:
import pandas as pd
import numpy as np
import plotly.express as px

In [53]:
cta_df = pd.read_parquet('../extract_ridership_data/output/cta_ridership.parquet')

In [54]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides
0,40350,UIC-Halsted,2001-01-01T00:00:00.000,U,273
1,41130,Halsted-Orange,2001-01-01T00:00:00.000,U,306
2,40760,Granville,2001-01-01T00:00:00.000,U,1059
3,40070,Jackson/Dearborn,2001-01-01T00:00:00.000,U,649
4,40090,Damen-Brown,2001-01-01T00:00:00.000,U,411
5,40590,Damen/Milwaukee,2001-01-01T00:00:00.000,U,870
6,40720,East 63rd-Cottage Grove,2001-01-01T00:00:00.000,U,391
7,41260,Austin-Lake,2001-01-01T00:00:00.000,U,399
8,40230,Cumberland,2001-01-01T00:00:00.000,U,788
9,41120,35-Bronzeville-IIT,2001-01-01T00:00:00.000,U,448


# Station features

In [55]:
cta_stations_df = pd.read_parquet('../extract_ridership_data/output/cta_stations.parquet')

In [56]:
cta_stations_df.head(10)

,stop_id,direction_id,stop_name,station_name,station_descriptive_name,map_id,ada,red,blue,g,...,p,y,pnk,o,location,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_vrxf_vc4k,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu
0,30162,W,18th (54th/Cermak-bound),18th,18th (Pink Line),40830,True,False,False,False,...,False,False,True,False,"{""latitude"":""41.857908"",""longitude"":""-87.66914...",8,14920,33,343,26
1,30161,E,18th (Loop-bound),18th,18th (Pink Line),40830,True,False,False,False,...,False,False,True,False,"{""latitude"":""41.857908"",""longitude"":""-87.66914...",8,14920,33,343,26
2,30022,N,35th/Archer (Loop-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,...,False,False,False,True,"{""latitude"":""41.829353"",""longitude"":""-87.68062...",26,14924,56,719,1
3,30023,S,35th/Archer (Midway-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,...,False,False,False,True,"{""latitude"":""41.829353"",""longitude"":""-87.68062...",26,14924,56,719,1
4,30213,N,35-Bronzeville-IIT (Harlem-bound),35th-Bronzeville-IIT,35th-Bronzeville-IIT (Green Line),41120,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",12,21194,1,25,9
5,30214,S,35-Bronzeville-IIT (63rd-bound),35th-Bronzeville-IIT,35th-Bronzeville-IIT (Green Line),41120,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",12,21194,1,25,9
6,30245,N,43rd (Harlem-bound),43rd,43rd (Green Line),41270,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.816462"",""longitude"":""-87.61902...",12,4301,4,162,9
7,30246,S,43rd (63rd-bound),43rd,43rd (Green Line),41270,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.816462"",""longitude"":""-87.61902...",12,4301,4,162,9
8,30210,S,47th (63rd-bound),47th,47th (Green Line),41080,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.809209"",""longitude"":""-87.61882...",12,21192,4,448,9
9,30237,N,47th (Howard-bound),47th,47th (Red Line),41230,True,True,False,False,...,False,False,False,False,"{""latitude"":""41.810318"",""longitude"":""-87.63094...",12,14924,3,189,9


In [57]:
stop_id_unique = set(cta_stations_df["map_id"])

mask_station_id = cta_df["station_id"].isin(stop_id_unique)
# True if everything is valid
all_stations_in_stations_df = mask_station_id.all()
print(all_stations_in_stations_df)

False


In [58]:
invalid_rows = cta_df[~mask_station_id]
print(invalid_rows['stationname'].value_counts())

stationname
Randolph/Wabash     6607
Madison/Wabash      6216
Washington/State    2953
Homan                 31
Name: count, dtype: int64


All of these are closed stations; see [here](https://www.chicago-l.org/stations/randolph-wabash.html) for Randolph/Wabash, 
[here](https://www.chicago-l.org/stations/madison-wabash.html) for Madison/Wabash, and [here](https://www.chicago-l.org/stations/washington-state.html) for Washington/State. That means we can safely merge the stations df onto the ridership df and have complete
station information for active stations.

In [59]:
# First, deduplicate the stations df since it seems to have one row per cardinal direction.
cta_stations_df_merge = cta_stations_df.groupby('map_id').first().reset_index()
cta_stations_df_merge = cta_stations_df_merge[['map_id', 'red', 'blue', 'g', 'brn', 'p', 'y', 'pnk', 'o']]

cta_df = cta_df.merge(
    cta_stations_df_merge,
    how='left',
    left_on='station_id',
    right_on='map_id'
).dropna(
    subset=['map_id']
)

In [61]:
# Create categorical line feature
conditions_cta_line = [
    cta_df['red'] == 1,
    cta_df['blue'] == 1,
    cta_df['g'] == 1,
    cta_df['brn'] == 1,
    cta_df['p'] == 1,
    cta_df['y'] == 1,
    cta_df['pnk'] == 1,
    cta_df['o'] == 1
]

choices_cta_line = [
    'red',
    'blue',
    'green',
    'brown',
    'purple',
    'yellow',
    'pink',
    'orange'
]

cta_df['line'] = np.select(conditions_cta_line, choices_cta_line, default='NA')

In [62]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,p,y,pnk,o,line
0,40350,UIC-Halsted,2001-01-01T00:00:00.000,U,273,40350,False,True,False,False,False,False,False,False,blue
1,41130,Halsted-Orange,2001-01-01T00:00:00.000,U,306,41130,False,False,False,False,False,False,False,True,orange
2,40760,Granville,2001-01-01T00:00:00.000,U,1059,40760,True,False,False,False,False,False,False,False,red
3,40070,Jackson/Dearborn,2001-01-01T00:00:00.000,U,649,40070,False,True,False,False,False,False,False,False,blue
4,40090,Damen-Brown,2001-01-01T00:00:00.000,U,411,40090,False,False,False,True,False,False,False,False,brown
5,40590,Damen/Milwaukee,2001-01-01T00:00:00.000,U,870,40590,False,True,False,False,False,False,False,False,blue
6,40720,East 63rd-Cottage Grove,2001-01-01T00:00:00.000,U,391,40720,False,False,True,False,False,False,False,False,green
7,41260,Austin-Lake,2001-01-01T00:00:00.000,U,399,41260,False,False,True,False,False,False,False,False,green
8,40230,Cumberland,2001-01-01T00:00:00.000,U,788,40230,False,True,False,False,False,False,False,False,blue
9,41120,35-Bronzeville-IIT,2001-01-01T00:00:00.000,U,448,41120,False,False,True,False,False,False,False,False,green


In [63]:
cta_df['line'].value_counts()

line
blue      288707
red       270329
green     246540
brown     198427
pink       99208
purple     90218
orange     63133
yellow     13926
Name: count, dtype: int64

# Date features 

## Year, month, and day

In [34]:
cta_df['date'] = pd.to_datetime(cta_df['date'])

cta_df['year'] = cta_df['date'].dt.year
cta_df['month'] = cta_df['date'].dt.month
cta_df['day'] = cta_df['date'].dt.day

## Day of week

In [35]:
cta_df['day_of_week_num'] = cta_df['date'].dt.weekday
cta_df['day_of_week_name'] = cta_df['date'].dt.day_name()

cta_df.head(10)

,station_id,stationname,date,daytype,rides,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,2001,1,1,0,Monday


# Bin rides into quartiles

In [36]:
cta_df['rides'] = cta_df['rides'].astype(int)

In [37]:
# Graph the distribution of ridership by year

# One year after COVID-19
fig_2024 = px.histogram(
    cta_df[cta_df['year'] == 2024],
    x="rides",
    nbins=30
)

fig_2024.show()

In [38]:
# Pick a pre-COVID-19 year
fig_2017 = px.histogram(
    cta_df[cta_df['year'] == 2017],
    x="rides",
    nbins=30
)

fig_2017.show()

In [39]:
cta_df[cta_df['year'] == 2017]['rides'].describe()

count    52713.000000
mean      3579.106729
std       3521.531320
min          0.000000
25%       1190.000000
50%       2522.000000
75%       4521.000000
max      28161.000000
Name: rides, dtype: float64

In [40]:
cta_df[cta_df['year'] == 2024]['rides'].describe()

count    52522.000000
mean      2060.416245
std       2014.531080
min          0.000000
25%        665.000000
50%       1417.500000
75%       2712.000000
max      21862.000000
Name: rides, dtype: float64

In [41]:
labels_ridership = ['low', 'medium', 'high']

cta_df['rides_quartile'] = (
    cta_df.groupby('year')['rides']
          .transform(lambda x: pd.qcut(x, 3, labels=labels_ridership, duplicates='drop'))
)

In [42]:
cta_df['rides_quartile'].value_counts()

rides_quartile
low       428610
high      428359
medium    428326
Name: count, dtype: int64

# Save the data

In [43]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,year,month,day,day_of_week_num,day_of_week_name,rides_quartile
0,40350,UIC-Halsted,2001-01-01,U,273,2001,1,1,0,Monday,low
1,41130,Halsted-Orange,2001-01-01,U,306,2001,1,1,0,Monday,low
2,40760,Granville,2001-01-01,U,1059,2001,1,1,0,Monday,low
3,40070,Jackson/Dearborn,2001-01-01,U,649,2001,1,1,0,Monday,low
4,40090,Damen-Brown,2001-01-01,U,411,2001,1,1,0,Monday,low
5,40590,Damen/Milwaukee,2001-01-01,U,870,2001,1,1,0,Monday,low
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,2001,1,1,0,Monday,low
7,41260,Austin-Lake,2001-01-01,U,399,2001,1,1,0,Monday,low
8,40230,Cumberland,2001-01-01,U,788,2001,1,1,0,Monday,low
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,2001,1,1,0,Monday,low


In [44]:
cta_df.tail(10)

,station_id,stationname,date,daytype,rides,year,month,day,day_of_week_num,day_of_week_name,rides_quartile
1285285,41480,Western-Brown,2025-08-31,U,1340,2025,8,31,6,Sunday,medium
1285286,41490,Harrison,2025-08-31,U,2756,2025,8,31,6,Sunday,high
1285287,41500,Montrose-Brown,2025-08-31,U,1022,2025,8,31,6,Sunday,medium
1285288,41510,Morgan-Lake,2025-08-31,U,3702,2025,8,31,6,Sunday,high
1285289,41660,Lake/State,2025-08-31,U,10951,2025,8,31,6,Sunday,high
1285290,41670,Conservatory,2025-08-31,U,558,2025,8,31,6,Sunday,low
1285291,41680,Oakton-Skokie,2025-08-31,U,250,2025,8,31,6,Sunday,low
1285292,41690,Cermak-McCormick Place,2025-08-31,U,1459,2025,8,31,6,Sunday,medium
1285293,41700,Washington/Wabash,2025-08-31,U,6586,2025,8,31,6,Sunday,high
1285294,41710,Damen-Lake,2025-08-31,U,659,2025,8,31,6,Sunday,low


In [45]:
cta_df.to_parquet('output/cta_ridership_with_features.parquet', engine='fastparquet', index=False)